In [ ]:
### Script for Pathway enrichment analysis on MOFA results; requires a prepared pathway set input dataset

#############################################
# Prerequisites - Load Libraries

In [ ]:
source('MS1_Functions.r')

In [ ]:
### Inform about execution start
popup_function_pos('06_Downstream_Pathways: Execution Started')

In [ ]:
source('MS0_Libraries.r')

In [ ]:
source('MS2_Plot_Config.r')

###############################################
# Preqrequisites Configurations & Parameters

In [ ]:
### Load the parameters that are set via the configuration files

In [ ]:
### Load configurations file
global_configs = read.csv('configurations/Data_Configs.csv', sep = ',')

In [ ]:
head(global_configs,3)

In [ ]:
data_path = global_configs$value[global_configs$parameter == 'data_path']

In [ ]:
data_path

In [ ]:
result_path = global_configs$value[global_configs$parameter == 'result_path']

In [ ]:
result_path

In [ ]:
### Load the configuration for the pathway enrichment from the config file

In [ ]:
pathway_configs = read.csv('configurations/06_Pathway_Configs.csv', sep = ',')
pathway_configs = pathway_configs[pathway_configs$mofa_result_name != '',]

In [ ]:
head(pathway_configs)

In [ ]:
### Generate the result data directory if it does not exist yet
if(!file.exists(paste0(result_path, '06_results'))){
    dir.create(file.path(paste0(result_path, '06_results')))
    }

# Define parameters 

In [ ]:
### Save values from loaded config file in variables

In [ ]:
## For the calculation of the pathway enrichment

In [ ]:
mofa_name = pathway_configs$mofa_result_name[1]   # mofa results that should be used

In [ ]:
factor_set = as.numeric(unlist(str_split(pathway_configs$factor_set[1], ',')))  # factors for which enrichment analysis should be executed

In [ ]:
factor_set

In [ ]:
length(factor_set)

In [ ]:
### Factor set needs to be at least first to factors if factor-set only one factor does not work

In [ ]:
if(length(factor_set) <= 1){
    if(factor_set <= 1){
        factor_set = c(1,2)}
    }
if(length(factor_set) <= 1){
    if(factor_set > 1){
        factor_set = 1:factor_set
        }
    }

In [ ]:
factor_set

In [ ]:
coverage_par = pathway_configs$coverage_par[1] # coverage parameter: how many of the genes of a pathway need to be included in the MOFA feature set for testing this pathway for enrichment

In [ ]:
coverage_par

In [ ]:
views_set = unlist(str_split( pathway_configs$types[1], ','))  # extract the views for which a view-specific pathway enrichment analysis should be executed

In [ ]:
views_set

In [ ]:
## Parameters for the visualization of pathways

In [ ]:
### Select pathways based on thresholds (like coverage, p-value, direction of enrichment)
coverage_plot = pathway_configs$coverage_plot[1]
p_value_cutoff_plot =pathway_configs$p_value_plot[1]
max_pathways =pathway_configs$max_pathways_plot[1]
select_enrichment = pathway_configs$enrichment_plot[1]

### Alternative: select pathways based on their specified names
pathway_selection_var =pathway_configs$pathway_selection[1]

In [ ]:
pathway_selection_var

In [ ]:
### For visualization define which genes should be ploted (need to be among the top x% of features for the Factor)
top_var_thres =pathway_configs$top_features_plot[1] # choose threshold of top x % of features of MOFA factor to take into account

In [ ]:
select_enrichment

In [ ]:
## Fixed parameters (may be modified here)

In [ ]:
## For enrichment calculation
use_statistic = "rank.sum" # which statistic to use to calcuate the enrichment; alternatives: mean.diff, rank.sum
use_test = 'parametric'  # which test to use to test the enrichment; alternatives: permutation, parametric, "cor.adj.parametric"
p_val_cutoff = 0.05
min_size = 5

# Load Data 

## Model Data

In [ ]:
### Load the trained MOFA Model

In [ ]:
model_name =  paste0("03_MOFA_MODEL_", mofa_name,'.hdf5')

In [ ]:
outfile = file.path( paste0(result_path, '/03_results/',  model_name) )

In [ ]:
outfile

In [ ]:
if(file.exists(outfile)){
    model <- load_model(outfile, verbose = TRUE)
    popup_function_pos(paste0('Loaded: ', model_name))
    } else {popup_function_neg(paste0('Error: ',result_path, '/03_results/ ', model_name, 'could not be loaded. Check whether the previous steps have been executed successfully')) } 

## Pathways

In [ ]:
### Load the pre-defined pathway set( needs to include the columns:
# ID (unique identifier of the pathway)
# gene : gene-symbol of the gene belonging to the pathway (will be matched to the MOFA features)
# pathway_name: a textual description of the pathway 

In [ ]:
path = paste0(data_path, 'pathways_hallmark_cancer.csv')
if(file.exists(path)){
    pathways =  read.csv(path)
    pathways$X = NULL 
    popup_function_pos(paste0('Loaded: ','pathways_hallmark_cancer.csv'))
    

} else{  popup_function_neg(paste0('Error: ','pathways_hallmark_cancer.csv', ' could not be loaded. Make sure the file is included within the specified input data folder'))
       print(paste0('Error: ','Prepared_Pathway_Data.csv', ' could not be loaded. Make sure the file is included within the specified input data folder'))
      
      
      }

In [ ]:
head(pathways,2)

# Prepare model data

## Extract the weights from the model

In [ ]:
### Get the feature weights from the model and prepare the format

In [ ]:
weights = get_weights(model, views = "all", factors = "all")
weight_data = data.frame()
for (i in names(weights)){
    data = data.frame(weights[[i]])
    data$type = i
    weight_data = rbind(weight_data,data)
}
weight_data$variable_name = rownames(weight_data)
weight_data$view <- weight_data$type 
#weight_data$gene = sapply(strsplit(weight_data$variable_name, "_"), "[", 3)
weight_data$gene <- sapply(strsplit(weight_data$variable_name, "__"), `[`, 2)
head(weight_data)

In [ ]:
## Transform to long format

In [ ]:
feature_weights_long = melt(weight_data)

In [ ]:
head(feature_weights_long,2)

## Adjust feature names in the model

In [ ]:
### Feature names should map to the genes in the pathway set (therefore the view component is  removed)

In [ ]:
head(features_names(model)[[1]] )

In [ ]:
model_conc = model ## save original model for later
for(i in names(features_names(model))){   
        features_names(model)[[i]] = with(feature_weights_long, gene[match(features_names(model)[[i]], variable_name)])
    }

In [ ]:
head(features_names(model)[[1]] )

## Create a MOFA model for the overall enrichment analysis across views

In [ ]:
# We need a model that has features of all views concatenated in a single view

In [ ]:
views <- names(features_names(model_conc))
tmp <- sapply(views, function(view) model_conc@intercepts[[view]]$group1)
names(tmp) <- NULL  
model_conc@intercepts[['complete']]$group1 = unlist(tmp)
model_conc@expectations$W[['complete']] = do.call(rbind, lapply(views, function(view) model_conc@expectations$W[[view]]))
model_conc@features_metadata = rbind(model_conc@features_metadata, 
                                     data.frame(feature = model_conc@features_metadata$feature, view="complete"))
model_conc@dimensions$D['complete'] = sum(model_conc@dimensions$D)
model_conc@data[['complete']]$group1 = do.call(rbind, lapply(views, function(view) model_conc@data[[view]]$group1))
model_conc@data_options$views = c(model_conc@data_options$views , 'complete')
model_conc@model_options$likelihoods['complete'] = 'gaussian'
model_conc@dimensions$M = length(views) + 1

model_conc@cache$variance_explained$r2_total$group1['complete'] = mean(model_conc@cache$variance_explained$r2_total$group1)
model_conc@cache$variance_explained$r2_per_factor$group1 = cbind(model@cache$variance_explained$r2_per_factor$group1, data.frame('complete' = rowMeans(model_conc@cache$variance_explained$r2_per_factor$group1)))

In [ ]:
model_conc

# Prepare pathway data

In [ ]:
### Prepare the pathway data to use for the enrichment

In [ ]:
head(pathways,2)

## Filter pathways out based on coverage

In [ ]:
## get all features included in the MOFA model
mofa_genes = data.frame(gene = unique(feature_weights_long$gene), is_feature = 1)

In [ ]:
head(mofa_genes,2)

In [ ]:
### Merge features within mofa model to pathways
feature_set = merge(pathways, mofa_genes, all.x = TRUE)  

In [ ]:
head(feature_set,2)

In [ ]:
### Remove pathways for which we have not a high amount of genes in our data (coverage_par)
filter = feature_set %>% group_by(ID, pathway_name) %>% summarise(gene_amount = n(),matched_amount = sum(!is.na(is_feature)),  coverage = sum(!is.na(is_feature)) / n()) %>% filter(coverage >=  coverage_par)

### Get the pathways that have been filtered out because of to low coverage
filtered_pathways = feature_set %>% group_by(ID, pathway_name) %>% summarise(gene_amount = n(),matched_amount = sum(!is.na(is_feature)),  coverage = sum(!is.na(is_feature)) / n()) %>% filter(coverage <  coverage_par)
nrow(filtered_pathways)  # amount of pathways that have een filtred out

In [ ]:
head(filter,2)

In [ ]:
feature_set = merge(feature_set, filter)

In [ ]:
# Overview pathways that have been excluded from testing due to low amount of matching genes
head(filtered_pathways,15) 

In [ ]:
### Remove NA entries for not mapped genes in feature set

In [ ]:
head(feature_set,2)

In [ ]:
feature_set = feature_set[!is.na(feature_set$is_feature),]

In [ ]:
nrow(feature_set)

In [ ]:
head(feature_set,2)

In [ ]:
### Save coverage for later usage
coverage_info = unique(feature_set[,c('ID', 'pathway_name', 'coverage')])

In [ ]:
### Get genes that are not mapped / and add a not mapped pathway to use them in the background later
# (this is for including all features that are in the MOFA model in the enrichment analysis, either as belonging to a pathway or only being part of the background set)

In [ ]:
non_pathway_genes = unique(mofa_genes$gene[!mofa_genes$gene %in% unique(feature_set$gene)])

In [ ]:
add_pathway = data.frame(ID = 'Background', pathway_name = 'Background', gene = non_pathway_genes, 
                         is_feature = 1, gene_amount = length(non_pathway_genes), matched_amount =  length(non_pathway_genes),
                         coverage = 1)

In [ ]:
feature_set = rbind(feature_set, add_pathway)

## Transform to binary matrix format

In [ ]:
### Transform the pathway feature set to a binary matrix format (1 indicating that a certain feature belongs to the pathway; 0 that it does not)

In [ ]:
head(feature_set,2)

In [ ]:
feature_set$pathway_id = paste0(feature_set$ID, '_', feature_set$pathway_name)

In [ ]:
feature_set$value = 1

In [ ]:
### Adjust names for overall approach (concatenate gene names with cell-types and match to pathways)

In [ ]:
feature_weights_long_mapped_join = unique(feature_weights_long[,c('gene', 'variable_name')])

In [ ]:
head(feature_weights_long_mapped_join,2)

In [ ]:
feature_set_all = merge(feature_set, feature_weights_long_mapped_join, by.x = 'gene', by.y = 'gene')

In [ ]:
nrow(feature_set_all)

In [ ]:
### Binarize feature set to use in type-specific enrichment

In [ ]:
## Based on gene-name for overall enrichment

feature_set = unique(feature_set[,c('pathway_id', 'gene', 'value')]) %>% dcast(pathway_id ~ gene, value.var = 'value')
feature_set[is.na(feature_set)]= 0

In [ ]:
head(feature_set,2)

In [ ]:
### Binarize feature set to use in overall enrichment

In [ ]:
feature_set_all = unique(feature_set_all[,c('pathway_id', 'variable_name', 'value')]) %>% dcast(pathway_id ~ variable_name, value.var = 'value')
feature_set_all[is.na(feature_set_all)] = 0

In [ ]:
head(feature_set_all,2) # feature set containing only genes

In [ ]:
## Adjust rownames and convert to matrix

In [ ]:
rownames(feature_set) = feature_set$pathway_id
rownames(feature_set_all) = feature_set_all$pathway_id

In [ ]:
feature_set$pathway_id = NULL
feature_set_all$pathway_id = NULL

In [ ]:
feature_set = as.matrix(feature_set)
feature_set_all = as.matrix(feature_set_all)

# Run pathway enrichment

## Per View

In [ ]:
### Filter views_set on relevant views with overlaps to pathway features

In [ ]:
views_set

In [ ]:
views_set = views_set[views_set %in% unique(str_replace(colnames(feature_set_all)[colSums(feature_set_all[rownames(feature_set_all) != 'Background_Background',]) > 5], '__.*', ''))]

In [ ]:
views_set

In [ ]:
## For each view seperately run the enrichment

In [ ]:
if(max(views_set) != ''){
    enrichment_result_types = run_enrichment_pathway(
        model = model, # MOFA Model
        factor_set = factor_set, # list of factors for which to run the enrichment
        views = views_set,
        use_statistic = use_statistic, # which statistic to use
        feature_set = feature_set, # Pathway Feature Set mapping; here use without type names
        min_size = min_size, # Min size of genes within a pathway
        use_test = use_test, # test used for calculating p-value
        p_val_cutoff = p_val_cutoff, # p-value cutoff used
        enrichment_result_p_val = data.frame())# dataset for saving result 
    } else{enrichment_result_types = ''}

In [ ]:
# Example of the resulting enrichment table

In [ ]:
if(is.data.frame(enrichment_result_types)){
    head(enrichment_result_types  %>% arrange(padj) ,5)}

## Across Views

In [ ]:
## Run an enrichment analysis across all the views

In [ ]:
#model_conc

In [ ]:
#data <- get_data(model_conc, views = "complete", as.data.frame = FALSE)[[1]]
#if (is.list(data)) { data <- Reduce(cbind, data) }
#data <- t(data)  # Features are columns

# Calculate variance for each feature
#variances <- apply(data, 2, function(x) var(x, na.rm = TRUE))
#problematic_features <- names(variances)[is.na(variances) | variances == 0]
#print(problematic_features)

In [ ]:
#data

In [ ]:
enrichment_result_all = run_enrichment_pathway(
    model = model_conc, # MOFA Model
    views = 'complete', # dimensions for which to run
    factor_set = factor_set, # list of factors for which to run the enrichment
    use_statistic = use_statistic, # which statistic to use
    feature_set = feature_set_all, # Pathway Feature Set mapping; use the concatenated one with cell-types
    min_size = min_size, # Min size of genes within a pathway
    use_test = use_test, # test used for calculating p-value
    p_val_cutoff = p_val_cutoff, # p-value cutoff used
    enrichment_result_p_val = data.frame()) # dataset for saving results

In [ ]:
head(enrichment_result_all %>% arrange(padj) ,5)

In [ ]:
enrichment_result_all

In [ ]:
### Combine both versions and save the result

In [ ]:
coverage_info

In [ ]:
if(is.data.frame(enrichment_result_types)){
    enrichment_result = rbind(enrichment_result_all, enrichment_result_types  )
    } else{enrichment_result = enrichment_result_all}

In [ ]:
enrichment_result$ID = str_replace(enrichment_result$pathway, '_.*', '')
enrichment_result$pathway_name = str_replace(enrichment_result$pathway, '.*_', '')

In [ ]:
enrichment_result$pathway = NULL

In [ ]:
enrichment_result = merge(enrichment_result, coverage_info)

In [ ]:
write.csv(enrichment_result, paste0(result_path, '/06_results/06_Pathway_Enrichment_' ,mofa_name, '.csv'), row.names = FALSE)

# Plot interesting pathways

In [ ]:
## Plot a subset of the enriched pathways (based on the parameters of the config file)

In [ ]:
pathway_selection_var  # specifies whether a specific pathway has been selected in the config file

In [ ]:
# specifies whether pathways will be displayed based on p-value and coverage thresholds specified in the config file
# or specifically selected pathways will be shown
if(pathway_selection_var == '' | is.na(pathway_selection_var) ){
    pathways_selection = enrichment_result[
        (enrichment_result$coverage > coverage_plot) & 
        (enrichment_result$enrichment == select_enrichment) &
        (enrichment_result$global_FDR < p_value_cutoff_plot),] %>%  arrange(padj) %>% group_by(variable) %>% top_n(max_pathways, -padj)
    } else{ pathways_selection = enrichment_result[enrichment_result$pathway_name %in% unlist(str_split(pathway_selection_var, ',')),]}

In [ ]:
## Select the genes of the pathway that have the highest weights on the selected factor

In [ ]:
### Get involved genes

## Define gene-set to merge
geneset_oi_pos_per_factor = feature_weights_long %>% group_by(variable) %>% arrange( desc(value),  .by_group = TRUE)  %>% top_frac(as.numeric(top_var_thres))
geneset_oi_pos_per_factor$direction = 'positive'
geneset_oi_neg_per_factor = feature_weights_long %>% group_by(variable) %>% arrange(desc(value),  .by_group = TRUE)  %>% top_frac(-as.numeric(top_var_thres))
geneset_oi_neg_per_factor$direction = 'negative'
geneset_oi = rbind(geneset_oi_pos_per_factor, geneset_oi_neg_per_factor)

colnames(geneset_oi) = c('type', 'variable_name','view',  'gene', 'variable', 'factor_value', 'factor_value_direction')
head(geneset_oi,2)

In [ ]:
### Merge genes belongig to pathway to enriched pathway sets

pathways_vis_genes = merge(pathways_selection, pathways) %>% mutate(pvalue=p, view_text=view, cluster=view, name=pathway_name) %>% dplyr::select(-view)

In [ ]:
### Add the feature weights to the corresponding genes from geneset_oi

In [ ]:
pathways_vis_genes = merge(pathways_vis_genes, geneset_oi, by.x = c('gene', 'variable'), by.y = c('gene','variable' ))

In [ ]:
### Summarise to get max/ mean factor value of each gene per pathway (remove cell-cluster/view/type dimension)  --> tBD max or mean (for max --> absolute value?)

pathways_vis_genes_summarized = pathways_vis_genes %>% group_by(gene ,variable,  pathway_name, ID, name, cluster ,view) %>% summarise(factor_value = mean(factor_value), enrichment_type = paste0(unique(enrichment), collapse = '&'), pvalue = min(pvalue))

In [ ]:
head(pathways_vis_genes_summarized,2)

In [ ]:
### Plot the pathways with summarized gene values

In [ ]:
pathways_sum_plot = list()

In [ ]:
for(i in unique(pathways_vis_genes_summarized$variable)){

    xlabel = xlab('Gene') 
    ylabel = ylab('Pathway')
    
    pathways_sum_plot[[i]] = ggplot(pathways_vis_genes_summarized[pathways_vis_genes_summarized$variable == i,], aes(gene,  pathway_name, fill= factor_value)) + 
        plot_config_heatmap + 
        geom_tile() + 
        scale_fill_gradient2(low = "#1D2ED8", mid = "white", high = "#D8911D", midpoint = 0)  + 
        scale_x_discrete(position = "top") +
        theme(axis.text.x = element_text(angle = 90), axis.title.x = element_blank(), axis.text.y = element_text(hjust = 0, vjust = 0.5)) +
        xlabel +
        ylabel +  scale_y_discrete(labels = label_wrap(50))+
        ggtitle(paste0(i, ' values of pathway genes (top ', top_var_thres  * 2 *100, '% of features)'))
    }

In [ ]:
options(repr.plot.width=20, repr.plot.height=10)
#pathways_sum_plot[[i]] +  scale_y_discrete(labels = label_wrap(25))

In [ ]:
## Plot factor values for all genes

In [ ]:
pathways_detail_plot = list()

In [ ]:
### Visualized the exact factor values of the genes
for(i in unique(pathways_vis_genes$variable)){
    # Specific Text Descriptions:
    xlabel = xlab('Gene') 
    ylabel = ylab('View')

    plot_data_cluster = unique(pathways_vis_genes[pathways_vis_genes$variable == i,c('gene', 'variable', 'type','view',  'variable_name', 'factor_value', 'factor_value_direction')])

    pathways_detail_plot[[i]] = ggplot(plot_data_cluster, aes(gene,  view, fill= factor_value)) + 
        plot_config_heatmap + 
        geom_tile() + 
        scale_fill_gradient2(low = "#1D2ED8", mid = "white", high ="#D8911D", midpoint = 0)  + 
        scale_x_discrete(position = "top") +
        theme(axis.text.x = element_text(angle = 90)) +
        xlabel +
        ylabel
}

In [ ]:
#pathways_detail_plot[[i]]

In [ ]:
### Combine plot and save

In [ ]:
combined_plot = list()

In [ ]:
combined_plot

In [ ]:
for(i in 1:length(pathways_detail_plot)){
    combined_plot[[i]] = ggarrange(pathways_sum_plot[[i]],
              pathways_detail_plot[[i]] + theme(axis.text.x = element_blank(), axis.title.x =element_blank()), align = 'v', ncol = 1)
}

In [ ]:
#combined_plot[[i]]

In [ ]:
## Save the plot

In [ ]:
figure_name = paste0( "FIG06_Pathways_and_Genes")

In [ ]:
# Sizes of the plot
width_par = 8.07
height_par = 4

In [ ]:
pdf(paste0('figures/06_figures/', figure_name, '_',   mofa_name, '.pdf'), width =width_par, height =height_par)

for(j in 1:length(combined_plot)){
        print( combined_plot[[j]])
        }
dev.off()   


In [ ]:
### Inform about execution finalization
popup_function_pos('06_Downstream_Pathways: Execution Finished')

In [ ]:
Sys.sleep(20)
popup_function_info('06_Downstream_Pathways')